# Phase 4: Build Training and Unseen Labeled Sets (Custom Targets)

This notebook builds:
- a training dataset with custom targets derived from the current labeled pool
- an unseen labeled dataset from all remaining labeled rows


In [1]:
from pathlib import Path
import json
import pandas as pd
import re

REPO_ROOT = Path.cwd().resolve()
DATA_ROOT = Path('/root/separate_volume')
if not DATA_ROOT.exists():
    DATA_ROOT = REPO_ROOT

RUN_ID = pd.Timestamp.now().strftime('split_%Y%m%d_%H%M%S')
INPUT_CSV = DATA_ROOT / 'datasets/labeled/annotator_a_llm.csv'
OUTPUT_DIR = DATA_ROOT / 'datasets/splits/runs' / RUN_ID
CURRENT_DIR = DATA_ROOT / 'datasets/splits/current'
OUTPUT_TRAIN = OUTPUT_DIR / 'train_labeled_331.csv'
OUTPUT_UNSEEN = OUTPUT_DIR / 'unseen_labeled_rest.csv'
OUTPUT_SUMMARY = OUTPUT_DIR / 'train_unseen_331_summary.json'

SEED = 42
MIN_TEXT_CHARS = 8
MAX_TEXT_CHARS = 1200
SINHALA_RATIO_MIN = 0.4

TRAIN_TARGETS = {
    'DISINFO': 11881,
    'HATE': 18930,
    'NORMAL': 28395,
}

BOILERPLATE_PATTERNS = [
    'like and share',
    'subscribe',
    'join our channel',
    'follow us',
    'whatsapp',
    'telegram',
]

INPUT_CSV, OUTPUT_TRAIN, OUTPUT_UNSEEN


(WindowsPath('D:/client-projects/sl-social-media-risk-analysis/annotation/workflow/current/annotator_a_llm.csv'),
 WindowsPath('D:/client-projects/sl-social-media-risk-analysis/datasets/splits/runs/split_20260316_232015/train_labeled_331.csv'),
 WindowsPath('D:/client-projects/sl-social-media-risk-analysis/datasets/splits/runs/split_20260316_232015/unseen_labeled_rest.csv'))

In [2]:
URL_ONLY_RE = re.compile(r'^(https?://\S+|www\.\S+|\S+\.com\S*|\S+\.lk\S*)+$', re.IGNORECASE)
REPEATED_CHAR_RE = re.compile(r'(.)\1{6,}', re.DOTALL)
NON_TEXT_RE = re.compile(r'[\W_]+', re.UNICODE)

def normalize_text(text: str) -> str:
    text = '' if pd.isna(text) else str(text)
    text = text.replace('\u200d', '')
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'(.)\1{3,}', r'\1\1', text)
    return text

def sinhala_ratio(text: str) -> float:
    letters = [ch for ch in text if ch.isalpha()]
    if not letters:
        return 0.0
    sinhala_count = sum(1 for ch in letters if '\u0D80' <= ch <= '\u0DFF')
    return sinhala_count / max(len(letters), 1)

def is_noise(text: str) -> bool:
    stripped = text.strip()
    if not stripped:
        return True
    if URL_ONLY_RE.match(stripped.replace(' ', '')):
        return True
    lower = stripped.lower()
    for pattern in BOILERPLATE_PATTERNS:
        if pattern in lower:
            return True
    letters_or_digits = ''.join(ch for ch in stripped if ch.isalnum())
    if not letters_or_digits:
        return True
    if REPEATED_CHAR_RE.search(stripped):
        return True
    return False

df = pd.read_csv(INPUT_CSV)
if 'candidate_id' not in df.columns or 'annotator_label' not in df.columns:
    raise ValueError('Required columns not found: candidate_id, annotator_label')

df['annotator_label'] = df['annotator_label'].fillna('').astype(str).str.strip().str.upper()
labeled = df[df['annotator_label'].isin(['NORMAL', 'HATE', 'DISINFO'])].copy()

text_col = 'clean_text' if 'clean_text' in labeled.columns else 'text'
if text_col not in labeled.columns:
    raise ValueError('Expected text or clean_text column')
labeled['text_norm'] = labeled[text_col].apply(normalize_text)
labeled = labeled[labeled['text_norm'].str.len() >= MIN_TEXT_CHARS]
labeled = labeled[labeled['text_norm'].str.len() <= MAX_TEXT_CHARS]
labeled = labeled[~labeled['text_norm'].apply(is_noise)].copy()
labeled['sinhala_ratio'] = labeled['text_norm'].apply(sinhala_ratio)
labeled = labeled[labeled['sinhala_ratio'] >= SINHALA_RATIO_MIN].copy()

# Keep one row per candidate_id.
# Do not drop duplicate normalized text here, because that removes too much
# valid DISINFO volume from the training pool.
labeled = labeled.drop_duplicates(subset=['candidate_id'], keep='first').copy()

# Shuffle deterministically before sampling.
labeled = labeled.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

available = labeled['annotator_label'].value_counts().to_dict()
for label, target in TRAIN_TARGETS.items():
    if available.get(label, 0) < target:
        raise ValueError(f'Not enough rows for {label}: available={available.get(label, 0)} target={target}')

train_parts = []
for label, target in TRAIN_TARGETS.items():
    part = labeled[labeled['annotator_label'] == label].sample(n=target, random_state=SEED)
    train_parts.append(part)

train_df = pd.concat(train_parts, ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
train_ids = set(train_df['candidate_id'].astype(str))
unseen_df = labeled[~labeled['candidate_id'].astype(str).isin(train_ids)].copy().reset_index(drop=True)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CURRENT_DIR.mkdir(parents=True, exist_ok=True)
train_df.to_csv(OUTPUT_TRAIN, index=False, encoding='utf-8')
unseen_df.to_csv(OUTPUT_UNSEEN, index=False, encoding='utf-8')

train_df.to_csv(CURRENT_DIR / 'train_labeled_331.csv', index=False, encoding='utf-8')
unseen_df.to_csv(CURRENT_DIR / 'unseen_labeled_rest.csv', index=False, encoding='utf-8')

summary = {
    'input_csv': str(INPUT_CSV),
    'outputs': {
        'train': str(OUTPUT_TRAIN),
        'unseen': str(OUTPUT_UNSEEN),
        'current_train': str(CURRENT_DIR / 'train_labeled_331.csv'),
        'current_unseen': str(CURRENT_DIR / 'unseen_labeled_rest.csv'),
    },
    'seed': SEED,
    'train_targets': TRAIN_TARGETS,
    'preprocess': {
        'min_text_chars': MIN_TEXT_CHARS,
        'max_text_chars': MAX_TEXT_CHARS,
        'dedupe_normalized_text': False,
        'sinhala_ratio_min': SINHALA_RATIO_MIN,
        'remove_url_only': True,
        'remove_emoji_only': True,
        'remove_boilerplate': True,
        'remove_repetition_spam': True,
    },
    'counts': {
        'input_rows': int(len(df)),
        'labeled_rows': int(len(labeled)),
        'train_rows': int(len(train_df)),
        'unseen_rows': int(len(unseen_df)),
    },
    'class_counts': {
        'train': train_df['annotator_label'].value_counts().to_dict(),
        'unseen': unseen_df['annotator_label'].value_counts().to_dict(),
    },
}
OUTPUT_SUMMARY.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary


{'input_csv': 'D:\\client-projects\\sl-social-media-risk-analysis\\annotation\\workflow\\current\\annotator_a_llm.csv',
 'outputs': {'train': 'D:\\client-projects\\sl-social-media-risk-analysis\\datasets\\splits\\runs\\split_20260316_232015\\train_labeled_331.csv',
  'unseen': 'D:\\client-projects\\sl-social-media-risk-analysis\\datasets\\splits\\runs\\split_20260316_232015\\unseen_labeled_rest.csv',
  'current_train': 'D:\\client-projects\\sl-social-media-risk-analysis\\datasets\\splits\\current\\train_labeled_331.csv',
  'current_unseen': 'D:\\client-projects\\sl-social-media-risk-analysis\\datasets\\splits\\current\\unseen_labeled_rest.csv'},
 'seed': 42,
 'train_targets': {'NORMAL': 2850, 'HATE': 2850, 'DISINFO': 950},
 'counts': {'input_rows': 66849,
  'labeled_rows': 63780,
  'train_rows': 6650,
  'unseen_rows': 57130},
 'class_counts': {'train': {'NORMAL': 2850, 'HATE': 2850, 'DISINFO': 950},
  'unseen': {'NORMAL': 47451, 'HATE': 9643, 'DISINFO': 36}}}

In [3]:
print("Training dataset:", OUTPUT_TRAIN)
print("Unseen dataset:", OUTPUT_UNSEEN)
print("Summary:", OUTPUT_SUMMARY)
print("\nTrain label counts:")
print(train_df["annotator_label"].value_counts())
print("\nUnseen label counts:")
print(unseen_df["annotator_label"].value_counts())


Training dataset: D:\client-projects\sl-social-media-risk-analysis\datasets\splits\runs\split_20260316_232015\train_labeled_331.csv
Unseen dataset: D:\client-projects\sl-social-media-risk-analysis\datasets\splits\runs\split_20260316_232015\unseen_labeled_rest.csv
Summary: D:\client-projects\sl-social-media-risk-analysis\datasets\splits\runs\split_20260316_232015\train_unseen_331_summary.json

Train label counts:
annotator_label
NORMAL     2850
HATE       2850
DISINFO     950
Name: count, dtype: int64

Unseen label counts:
annotator_label
NORMAL     47451
HATE        9643
DISINFO       36
Name: count, dtype: int64
